# GPT-4 Zero-Shot — Drug Reviews

Reproducible zero-shot evaluation from *Zero-Shot vs. Fine-Tuned* (structured prompts, constraint-based parsing, temperature=0). Requires `OPENAI_API_KEY`.

Legacy T5 zero-shot notebooks are in `archieve/`.


In [1]:
import os
import sys
from dotenv import load_dotenv
load_dotenv()  # automatically finds .env in parent dirs
PROJECT_ROOT = os.getcwd()
if not os.path.isdir(os.path.join(PROJECT_ROOT, 'zeroshot')):
    PROJECT_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.environ.setdefault('WANDB_DISABLED', 'true')


'true'

In [2]:
!pip install scikit-learn 

In [ ]:
from zeroshot.classifier import ZeroShotClassifier
from zeroshot.evaluate import evaluate_zero_shot, load_split_df, resolve_base_dir
from zeroshot.labels import DATASET_CONFIGS

CONFIG_KEY = 'drug_review'
BASE_DIR = resolve_base_dir()
MAX_SAMPLES = int(os.getenv('ZEROSHOT_MAX_SAMPLES', '0')) or None  # cap for dry runs

# gpt-4o-mini is cheaper for development; use gpt-4 for paper replication
MODEL = os.getenv('ZEROSHOT_MODEL', 'gpt-4o-mini')
BACKEND = os.getenv('ZEROSHOT_BACKEND', 'openai')  # or mistral_local

classifier = ZeroShotClassifier(model=MODEL, backend=BACKEND, temperature=0.0, max_tokens=10)
config = DATASET_CONFIGS[CONFIG_KEY]
print(f'Dataset: {config.name} | Model: {MODEL} | Base: {BASE_DIR}')

In [4]:
BASE_DIR

'/Users/kirthi/Documents/UCBerkeley/kirthi_portfolio/MIDS/Academic_Projects/Medical_NLP_Zeroshot_vs_Finetune_v2/'

In [5]:
val_df = load_split_df(config, 'validation', BASE_DIR)
test_df = load_split_df(config, 'test', BASE_DIR)
print("..",val_df)
print(f'Validation: {len(val_df)} | Test: {len(test_df)}')
print(val_df[config.label_column].value_counts().head())


..                               drugName                       condition  \
0              amoxicillin clavulanate   skin or soft tissue infection   
1                               gianvi  premenstrual dysphoric disorde   
2                              liletta                   birth control   
3     ethinyl estradiol levonorgestrel                   birth control   
4       ethinyl estradiol norgestimate                   birth control   
...                                ...                             ...   
5536                            mirena       abnormal uterine bleeding   
5537    drospirenone ethinyl estradiol                   birth control   
5538                       liraglutide                diabetes, type 2   
5539    ethinyl estradiol norgestimate                   birth control   
5540                          fiorinal                        migraine   

                                                 review  \
0     this med was given as a result of a deep go

In [6]:
val_results = evaluate_zero_shot(
    val_df,
    CONFIG_KEY,
    classifier,
    dataset_name='validation',
    max_samples=MAX_SAMPLES,
    results_dir=os.path.join(PROJECT_ROOT, 'Results'),
)


Zero-shot inference: 100%|██████████| 5541/5541 [55:11<00:00,  1.67it/s]   



ZERO-SHOT EVALUATION — DRUG REVIEWS (validation)
Model: gpt-4.1-mini | Backend: openai
Valid pairs: 5541/5541
Accuracy:     0.5335
F1 (macro):   0.4800
F1 (weighted): 0.5679
Precision:    0.4989
Recall:       0.5133

Classification report:
               precision    recall  f1-score   support

VERY_NEGATIVE       0.72      0.66      0.69       940
     NEGATIVE       0.24      0.50      0.32       409
      NEUTRAL       0.28      0.34      0.31       523
     POSITIVE       0.32      0.54      0.40      1014
VERY_POSITIVE       0.94      0.53      0.68      2655

     accuracy                           0.53      5541
    macro avg       0.50      0.51      0.48      5541
 weighted avg       0.67      0.53      0.57      5541


Saved confusion matrix: /Users/kirthi/Documents/UCBerkeley/kirthi_portfolio/MIDS/Academic_Projects/Medical_NLP_Zeroshot_vs_Finetune_v2/Results/zeroshot_confusion_matrix_drug_review_validation_openai.png
Saved predictions: /Users/kirthi/Documents/UCBerkeley/kir

In [7]:
test_results = evaluate_zero_shot(
    test_df,
    CONFIG_KEY,
    classifier,
    dataset_name='test',
    max_samples=MAX_SAMPLES,
    results_dir=os.path.join(PROJECT_ROOT, 'Results'),
)


Zero-shot inference: 100%|██████████| 22162/22162 [13:19:12<00:00,  2.16s/it]      



ZERO-SHOT EVALUATION — DRUG REVIEWS (test)
Model: gpt-4.1-mini | Backend: openai
Valid pairs: 22162/22162
Accuracy:     0.5375
F1 (macro):   0.4884
F1 (weighted): 0.5697
Precision:    0.5048
Recall:       0.5234

Classification report:
               precision    recall  f1-score   support

VERY_NEGATIVE       0.72      0.68      0.70      3760
     NEGATIVE       0.26      0.52      0.34      1634
      NEUTRAL       0.30      0.36      0.33      2093
     POSITIVE       0.32      0.53      0.40      4057
VERY_POSITIVE       0.92      0.53      0.67     10618

     accuracy                           0.54     22162
    macro avg       0.50      0.52      0.49     22162
 weighted avg       0.67      0.54      0.57     22162


Saved confusion matrix: /Users/kirthi/Documents/UCBerkeley/kirthi_portfolio/MIDS/Academic_Projects/Medical_NLP_Zeroshot_vs_Finetune_v2/Results/zeroshot_confusion_matrix_drug_review_test_openai.png
Saved predictions: /Users/kirthi/Documents/UCBerkeley/kirthi_portfo

## Optional: Mistral-7B-Instruct (local)

Set `ZEROSHOT_BACKEND=mistral_local` and run on a GPU. Uses 4-bit quantization by default.
